In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

In [12]:
df = pd.read_csv('bpl_properties_post_feature_selection.csv')

In [13]:
df.head()

,colony,property_type,Total Floors,Property Age,bedrooms,bathrooms,balconies,built_up_area,servant room,store room,furnishing_type,Overlooking_Categories,luxury_category,price
0,10.0,1.0,1.0,2.0,7,5,0,2800.0,0,0,1,1.0,2.0,105.0
1,22.0,0.0,2.0,1.0,3,3,1,1567.2,0,0,1,1.0,2.0,85.0
2,20.0,0.0,3.0,1.0,2,2,3,957.0,0,0,1,1.0,1.0,25.0
3,0.0,1.0,2.0,2.0,4,5,1,2400.0,0,0,0,3.0,1.0,65.0
4,13.0,1.0,3.0,1.0,6,6,2,4000.0,1,1,2,3.0,1.0,650.0


In [14]:
# one hot encode -> colony, Property Age, furnishing type, luxury category, Overlooking_Categories.

In [15]:
X = df.drop(columns=['price'])
y = df['price']

In [16]:
columns_to_encode = ['colony', 'Property Age', 'furnishing_type', 'luxury_category', 'Overlooking_Categories']

In [17]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [18]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['property_type','Total Floors', 'bedrooms', 'bathrooms','balconies', 'built_up_area', 'servant room', 'store room']),
        ('cat', OneHotEncoder(drop='first'), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [19]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [20]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [21]:
scores.mean()

0.6902907685742974

In [22]:
scores.std()

0.06564244250937393

In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [24]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['property_type',
                                                   'Total Floors', 'bedrooms',
                                                   'bathrooms', 'balconies',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['colony', 'Property Age',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'Overlooking_Categories'])])),
                ('regressor', LinearRegression())])

In [25]:
y_pred = pipeline.predict(X_test)

In [26]:
y_pred = np.expm1(y_pred)

In [27]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test),y_pred)

40.68511365053671

# SVR

In [35]:
# Creating a pipeline
pipeline1 = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', SVR(kernel='rbf'))
])

In [36]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline1, X, y_transformed, cv=kfold, scoring='r2')

In [37]:
scores.mean()

0.755981590865021

In [38]:
scores.std()

0.044177703364518824